# Exercise 3 — Resume Field Extractor

## Objective
Extract only the requested fields from resume text.

## Inputs
- resume_text
- fields_to_extract

## Expected Output
Strict JSON containing only the requested fields, with no additional explanation.

In [1]:
%pip install groq python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: C:\Users\ARSHIYA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-120b"

print("Groq client initialized successfully.")

Groq client initialized successfully.


In [3]:
resume_text = """
Rahul Sharma
Email: rahul.sharma@example.com
Phone: 9876543210

Education:
Bachelor of Computer Applications

Skills:
Python, Java, SQL, HTML, CSS

Experience:
Fresher with academic project experience.
"""

fields_to_extract = [
    "Name",
    "Email",
    "Phone",
    "Education",
    "Skills",
    "Experience"
]

print("Resume loaded.")
print("Fields to extract:", fields_to_extract)

Resume loaded.
Fields to extract: ['Name', 'Email', 'Phone', 'Education', 'Skills', 'Experience']


In [4]:
prompt_template = """
You are a resume information extraction assistant.

Extract only the fields requested from the resume below.

Rules:
1. Return a valid JSON object only.
2. Include every requested field and no other fields.
3. Use null when a requested field is not present.
4. Do not guess or invent information.
5. Preserve extracted information accurately.
6. Use a JSON array for Skills if multiple skills are found.
7. Do not include Markdown fences or explanations.

Requested fields:
{fields_to_extract}

Resume:
{resume_text}
"""

prompt = prompt_template.format(
    fields_to_extract=json.dumps(fields_to_extract),
    resume_text=resume_text
)

print("Prompt created successfully.")

Prompt created successfully.


In [5]:
def extract_resume_fields(resume_text, fields_to_extract):
    if not resume_text.strip():
        raise ValueError("Resume text cannot be empty.")

    if not fields_to_extract:
        raise ValueError("Provide at least one field to extract.")

    prompt = prompt_template.format(
        fields_to_extract=json.dumps(fields_to_extract),
        resume_text=resume_text
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract resume information faithfully. "
                    "Return only valid JSON. Never invent missing values."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_completion_tokens=1000
    )

    content = response.choices[0].message.content

    if not content or not content.strip():
        raise ValueError("The model returned an empty response.")

    result = json.loads(content)

    if not isinstance(result, dict):
        raise ValueError("The response must be a JSON object.")

    if set(result.keys()) != set(fields_to_extract):
        raise ValueError(
            "JSON keys do not match requested fields. "
            f"Expected: {fields_to_extract}; received: {list(result.keys())}"
        )

    return result

In [6]:
extracted_data = extract_resume_fields(
    resume_text,
    fields_to_extract
)

print(json.dumps(extracted_data, indent=4, ensure_ascii=False))

{
    "Name": "Rahul Sharma",
    "Email": "rahul.sharma@example.com",
    "Phone": "9876543210",
    "Education": "Bachelor of Computer Applications",
    "Skills": [
        "Python",
        "Java",
        "SQL",
        "HTML",
        "CSS"
    ],
    "Experience": "Fresher with academic project experience."
}


In [7]:
assert isinstance(extracted_data, dict), "Output must be a JSON object."

assert set(extracted_data.keys()) == set(fields_to_extract), (
    "Output must contain exactly the requested fields."
)

for field in fields_to_extract:
    value = extracted_data[field]
    assert value is None or isinstance(value, (str, list, int, float, bool)), (
        f"Invalid JSON value for field: {field}"
    )

print("Validation passed: valid JSON object with exactly the requested fields.")

Validation passed: valid JSON object with exactly the requested fields.


In [8]:
test_resume = """
Name: Ananya Patil
Skills: Python, SQL
"""

test_fields = ["Name", "Email", "Skills"]

test_result = extract_resume_fields(
    test_resume,
    test_fields
)

print(json.dumps(test_result, indent=4, ensure_ascii=False))

assert set(test_result.keys()) == set(test_fields)
assert test_result["Email"] is None, (
    "Missing email should be represented as null."
)

print("Missing-field test passed.")

{
    "Name": "Ananya Patil",
    "Email": null,
    "Skills": [
        "Python",
        "SQL"
    ]
}
Missing-field test passed.


In [9]:
print("""
Conclusion:
The Resume Field Extractor uses a language model through the Groq API
to extract requested information from resume text. It returns a JSON
object containing only the requested fields and uses null for missing
information. The notebook validates JSON structure and tests a
resume with a missing field.
""")


Conclusion:
The Resume Field Extractor uses a language model through the Groq API
to extract requested information from resume text. It returns a JSON
object containing only the requested fields and uses null for missing
information. The notebook validates JSON structure and tests a
resume with a missing field.

